# How to use CBMViz - SQLite Database Operations

In [ ]:
import os
import cbmviz

CBMViz saves generated layouts internally as graph objects but they are not convenient for the user as their manipulation requires knowledge of the underlying Tulip API. Therefore, graphs are also stored in an SQLite database after creation. 

This makes CBMViz layouts compatible with a wide range of other visualization software since parts of the layout such as nodes and edges can be obtained as simple lists by querying the database.

In [ ]:
# run CBMViz core pipeline to create a layout
global_ops = cbmviz.cfg_default
sbml = os.path.join("TestModels", "iIT341.xml")

graph, graph_props = cbmviz.createInitialGraph("initialGraph")
cmod, fba_results, graph_registry, cDir, output_path = cbmviz.layoutModel(global_ops, sbml, graph, graph_props)
cbmviz.colorGraphs(global_ops, fba_results, graph_registry, output_path, "iIT341")
cbmviz.exportGraph(output_path, graph_registry, global_ops, cDir, "iIT341")

After creating a layout it can be stored in an SQLite database by runing the function **buildSQLiteDatabase** which takes in a graph object (stored in graph_registry) and the path of the output directory.

In [ ]:
# create SQLite db to store a graph object
db = cbmviz.buildSQLiteDatabase(graph_registry['output_whole'][0], output_path)

The resulting database will have five tables:
    
- *graph_nodes*: every row corresponds to a node, each column contains a node property
- *graph_edges*: every row corresponds to an edge, each column contains an edge property
- *node_connectivity*: every row corresponds to a node, the first column contains node ids and the second column contains lists of connected edges
- *edge connectivity*: every row corresponds to an edge, the first column contains edge ids, the second column contains source node ids and third column contains target node ids
- *edge color*: every row corresponds to an edge, the first column contains edge ids, subsequent columns contain color sets which are tuples of the flux value and pickled RGB color tuples. 

Data can be retrieved from the database by runing **retrieveDataFromDatabaseTable**. It returns a list of row sublists as well as a list of column headers.

In [ ]:
# retrieve graph data from SQL database
db_graph_node_list, node_col_headers = cbmviz.retrieveDataFromDatabaseTable(db, 'graph_nodes')
db_graph_edge_list, edge_col_headers = cbmviz.retrieveDataFromDatabaseTable(db, 'graph_edges')
db_graph_edgeconn_list, edgeconn_col_headers = cbmviz.retrieveDataFromDatabaseTable(db, 'edge_connectivity')

# Examples

1.) Rebuild a graph object from the database

In [ ]:
# SQLite --> CBMViz
db_graph = cbmviz.buildTulipGraphFromSQL(db_graph_node_list, db_graph_edge_list, db_graph_edgeconn_list,
                                  node_col_headers, edge_col_headers, "graph_from_DB", output_path)
# export rebuilt graph from database
db_graph_dict = cbmviz.createGraphDictFromGraphObj(db_graph)
cbmviz.exportGraph(output_path, db_graph_dict, global_ops, cDir, "iIT341")

2.) Use queried data to build a graph with NetworkX

In [ ]:
# access SQLite database
db_graph_node_list, node_col_headers = cbmviz.retrieveDataFromDatabaseTable(db, 'graph_nodes')
db_graph_edgeconn_list, edgeconn_col_headers = cbmviz.retrieveDataFromDatabaseTable(db, 'edge_connectivity')

# select only Glycolysis nodes
db_glyc_node_list = list()
for sublist in db_graph_node_list:
    if sublist[node_col_headers.index("pathway")] == "Glycolysis":
        db_glyc_node_list.append(sublist[node_col_headers.index("node_id")])


# use NetworkX to build a graph containing only Glycolysis nodes
import networkx as nx

G = nx.Graph()
for sublist in db_graph_edgeconn_list:
    for node_id in db_glyc_node_list:
        if str(node_id) in sublist[edgeconn_col_headers.index("source_node_id")]:
            G.add_edge(node_id, sublist[edgeconn_col_headers.index("target_node_id")])
        elif str(node_id) in sublist[edgeconn_col_headers.index("target_node_id")]:
            G.add_edge(sublist[edgeconn_col_headers.index("source_node_id")], node_id)
            
# embed NetworkX graph in jupyter notebook
from nxpd import draw

draw(G, show='ipynb')

Since this is supposed to be a small test example the NetworkX code was kept brief and as such resulting graph lacks many features of the input layout.